# Session 3 · Robo-Advisor Lab (code track)

*Data-driven Investment Strategies, IESE MBA. Educational use only; not investment advice.*

A one-line spreadsheet backtest of a robo portfolio (weights × daily returns, summed) is a portfolio rebalanced **every day, for free, with no fee**. This notebook runs the same idea through the course engine, which rebalances by calendar or when weights drift, pays trading costs on what it trades, charges the robo's fee layer, and never treats a missing price as a 0% return.

You direct; the engine computes. Change the inputs in the cells marked **⟵ edit** and re-run.

In [ ]:
# Run once. In Colab this clones the course repo (public) and puts the engine on the path.
import os, sys
if not os.path.exists('dis_lab'):
    !git clone -q https://github.com/jose-suarezlledo/dis_lab.git
sys.path.insert(0, 'dis_lab')
import pandas as pd, numpy as np
from engine import data, metrics as m, robo, fx
from engine.portfolio import drift_band_portfolio
print('engine ready; snapshot prices to', data.load_manifest()['last_date'].max())

## 1. Your profile
The reference questionnaire scores three things: **tolerance** (feelings), **capacity** (facts) and **need** (what the goal requires). The profile is the lower of tolerance and capacity. Answer with the index of the option (0 = first).

In [ ]:
for q in robo.QUESTIONS:
    print(f'[{q.dimension}] {q.id}: {q.text}')
    for i, (label, score) in enumerate(q.options):
        print(f'    {i}: {label}  (score {score})')

answers = dict(loss=2, swings=2, experience=1, horizon=3, share=2, income=2, need=1)   # ⟵ edit
rp = robo.risk_profile(answers)
print('\n', rp.note)

## 2. The model portfolio
A robo keeps one table and maps you to a row. Here is the engine's table (UCITS ETFs in euros).

In [ ]:
profile = rp.profile          # ⟵ or set a number 1-10 to test any row
weights = robo.model_weights(profile)
display((robo.MODEL_PORTFOLIOS * 100).round(1))
print('profile', profile, '->', {k: round(v, 3) for k, v in weights.items()}, '| equity share', round(robo.equity_share(weights), 2))

## 3. Backtest: the Excel way vs a real engine
Same weights, same prices, same window (Dec 2015 to Jan 2025, when Indexa's published series ends). Four versions:
1. **The Excel way**: rebalanced every day at zero cost, no fee (`drift_band_portfolio` with band 0, checked daily).
2. **Monthly rebalancing**, 10 bps per trade, no fee.
3. **Drift band** (reset only when a weight is 5 points off target, checked monthly), 10 bps, no fee.
4. **The client's experience**: drift band, 10 bps, plus the robo's fee layer (0.43% a year on top of the funds' TERs).

In [ ]:
FEE = 0.0043      # ⟵ edit: advisory + custody layer per year (fund TERs are already inside the prices)
COST = 10         # ⟵ edit: one-way trading cost in bps
px = data.load_prices(list(weights)).loc['2015-12-31':'2025-01-31'].dropna()
runs = {
    'Excel way (daily, free, no fee)': drift_band_portfolio(px, weights, band=0.0, check='daily', cost_bps=0).nav,
    'Monthly, 10 bps, no fee': robo.robo_backtest(px, weights, fee=0.0, rebalance='monthly', cost_bps=COST).nav,
    'Drift band 5 pts, 10 bps, no fee': robo.robo_backtest(px, weights, fee=0.0, rebalance='band', band=0.05, cost_bps=COST).nav,
    'Drift band + fee layer': robo.robo_backtest(px, weights, fee=FEE, rebalance='band', band=0.05, cost_bps=COST).nav,
}
ix = data.load_indexa('10-100k€', freq=None)
runs[f'Indexa profile {profile} (published)'] = ix[profile][ix.index.dayofweek < 5]
levels = m.to_frequency(m.align(*[s.rename(k) for k, s in runs.items()]), 'monthly')
rf = data.risk_free_rate(levels.index[0], levels.index[-1], data.EURO_RF_SERIES)
tab = m.summary_table(levels, rf=rf).loc[['cagr', 'vol', 'sharpe', 'max_dd', 'tuw_days', 'worst_year']]
display(tab.T.style.format({'cagr': '{:.2%}', 'vol': '{:.2%}', 'sharpe': '{:.2f}', 'max_dd': '{:.1%}', 'worst_year': '{:.1%}'}))
(levels / levels.iloc[0] * 100).plot(figsize=(10, 5), title='Growth of 100, monthly');

**Read the table.** The gap between the Excel way and the client's experience is not return: it is trading costs and fees the spreadsheet never charged. The gap to Indexa's own profile is mostly the asset mix (Indexa holds more European and Japanese equity, inflation-linked and hedged US bonds), not fees: check it by changing the weights.

## 4. The euro investor: three views of world equity
Local (USD), unhedged (EUR) and hedged (EUR, synthetic: local return plus the EUR−USD short-rate carry).

In [ ]:
v = fx.currency_views(data.load_prices('URTH')['URTH'].loc['2015-12-31':]).resample('ME').last()
display(m.summary_table(v).loc[['cagr', 'vol', 'max_dd', 'worst_year']].T.style.format('{:.2%}'))
yr = pd.DataFrame({c: m.calendar_year_returns(v[c])['return'] for c in v})
display(yr.style.format('{:+.1%}'))

## 5. Goals: one profile for everything, or one bucket per goal?
Same money, same contributions, same fee, same bootstrapped history. The profile-based pot pays each goal when it comes due; each goal-based bucket is judged on its own.

In [ ]:
menu = ['IWDA.AS', 'EUNH.DE', 'XEON.DE']
mr = robo.monthly_returns_from_prices(data.load_prices(menu))
def bucket(p):
    if p == 0: return {'IWDA.AS': 0.0, 'EUNH.DE': 0.0, 'XEON.DE': 1.0}
    row = robo.MODEL_PORTFOLIOS.loc[p]
    return {'IWDA.AS': float(row['IWDA.AS'] + row['IS3N.DE']), 'EUNH.DE': float(row['EUNH.DE'] + row['XG7S.DE']), 'XEON.DE': float(row['XEON.DE'])}
goals = [robo.Goal('House deposit (3 y)', 62_000, 60_000, 3, bucket(0)),                       # ⟵ edit
         robo.Goal('Retirement (30 y)', 38_000, 500_000, 30, bucket(9), contribution=300)]     # ⟵ edit
sim = robo.goal_simulation(mr, goals, profile_weights=bucket(6), n_sims=3000, fee=0.005, seed=7)
display(sim.table.style.format({'p_met': '{:.0%}', 'median_wealth': '€{:,.0f}', 'p5_wealth': '€{:,.0f}', 'target': '€{:,.0f}'}))
print('all goals met:', {k: f'{v:.0%}' for k, v in sim.p_all_met.items()}, '| equity at start:', {k: f'{v:.0%}' for k, v in sim.equity_at_start.items()})

**Caveats you must state in a memo:** the euro history is 17 years and mostly a bull market (try the US series since 1987: `VFINX`, `VBMFX`); nominal euros; no taxes; the model table was built in 2026 knowing how these assets did.